In [1]:
# ============================================================
# RAG CONFIGURATION EVALUATION
# ============================================================
#
# PURPOSE
# -------
# Evaluate the 9 summaries produced by the frozen 3 x 3 RAG
# configuration experiment.
#
# The experimental configurations combine:
#
#   3 chunking strategies:
#       - Whole note
#       - Fixed-size
#       - Section-aware
#
#   x
#
#   3 embedding approaches:
#       - Gemini
#       - BGE
#       - MedCPT
#
# All summaries were generated using:
#
#   Generator: GPT-5.6-Luna
#   Retrieval: Top-20 chunks
#   Evidence ordering: chronological after retrieval
#
#
# EVALUATION STAGES
# -----------------
# Eval 1 — Coverage
#     Measures how much clinically relevant source information
#     is represented in the generated summary.
#
# Eval 2 — TF-IDF similarity
#     Measures lexical similarity between the generated summary
#     and the original clinical record.
#
# Eval 3 — Clinical LLM evaluation
#     Evaluates clinically important properties that lexical
#     metrics alone cannot reliably capture.
#
#
# IMPORTANT
# ---------
# This notebook DOES NOT regenerate RAG summaries.
#
# It loads the frozen summaries produced by:
#     01_configuration_summaries.ipynb
#
# This keeps generation and evaluation separate and prevents
# accidental additional LLM generation calls.
# ============================================================

In [17]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
# Project paths
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
# ...existing code...

In [10]:
# ============================================================
# LOAD FROZEN RAG SUMMARIES
# ============================================================
#
# These are the 9 summaries generated in the configuration
# experiment. They are treated as fixed inputs throughout this
# evaluation notebook.
# ============================================================

SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_config_summaries_gpt56luna_top20.json"
)

with SUMMARY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    rag_results = json.load(file)

print("Loaded configurations:", len(rag_results))

for configuration in rag_results:
    print("-", configuration)

Loaded configurations: 9
- Whole + Gemini
- Whole + BGE
- Whole + MedCPT
- Fixed + Gemini
- Fixed + BGE
- Fixed + MedCPT
- Section + Gemini
- Section + BGE
- Section + MedCPT


In [70]:
# ============================================================
# OPENAI EVALUATOR SETUP
# ============================================================

import os

from openai import OpenAI

EVALUATOR_MODEL = "gpt-5.6-luna"

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set.")

openai_client = OpenAI(
    api_key=api_key
)

print("Client class:", type(openai_client).__name__)
print("Client module:", type(openai_client).__module__)
print("Evaluator model:", EVALUATOR_MODEL)

Client class: OpenAI
Client module: openai
Evaluator model: gpt-5.6-luna


In [11]:
# ============================================================
# EVALUATION 1 — REFERENCE-FACT COVERAGE
# ============================================================
#
# PURPOSE
# -------
# Measure how completely each generated RAG summary captures
# the clinically important information contained in the
# complete source clinical record.
#
# Rather than relying on word overlap, coverage is evaluated
# against a frozen checklist of clinically important reference
# facts derived from the complete patient record.
#
#
# SCORING
# -------
# Each reference fact receives:
#
#   2 = Fully captured
#       The clinically important meaning is represented.
#
#   1 = Partially captured
#       The core concept is present, but meaningful clinical
#       information from the reference fact is missing.
#
#   0 = Absent
#       The clinical information is not represented.
#
#
# WHY THIS EVALUATION IS SEPARATE
# -------------------------------
# This metric evaluates INFORMATION COVERAGE only.
#
# It does NOT determine whether information in the summary is
# factually supported by the source record. Unsupported claims
# and other clinical-quality properties are evaluated
# separately in later evaluations.
#
#
# EXPERIMENTAL CONTROL
# --------------------
# - The SAME frozen reference facts are used for all 9
#   RAG configurations.
#
# - The SAME evaluator model and prompt are used for all
#   configurations.
#
# - The 9 generated summaries remain frozen and are not
#   regenerated in this notebook.
# ============================================================

In [47]:
# ============================================================
# FROZEN REFERENCE-FACT CHECKLIST
# ============================================================
#
# IMPORTANT:
# These reference facts are copied unchanged from the original
# RAG configuration experiment.
#
# They must remain fixed across all 9 configurations so that
# every summary is evaluated against the same clinical
# information target.
# ============================================================

In [48]:
reference_facts = [

    # ---------- Presentation ----------
    {
        "id": "F01",
        "category": "presentation",
        "fact": "Patient presented with progressive exertional breathlessness."
    },
    {
        "id": "F02",
        "category": "presentation",
        "fact": "Patient was hypoxic at presentation, with SpO2 89% on room air."
    },

    # ---------- Diagnosis / investigations ----------
    {
        "id": "F03",
        "category": "diagnosis",
        "fact": "Chronic thromboembolic pulmonary hypertension (CTEPH) was diagnosed."
    },
    {
        "id": "F04",
        "category": "investigation",
        "fact": "CT pulmonary angiography confirmed chronic thromboembolic disease."
    },
    {
        "id": "F05",
        "category": "investigation",
        "fact": "Echocardiography demonstrated severe pulmonary hypertension with right ventricular strain."
    },
    {
        "id": "F06",
        "category": "investigation",
        "fact": "V/Q scanning demonstrated mismatched perfusion defects consistent with CTEPH."
    },
    {
        "id": "F07",
        "category": "investigation",
        "fact": "NT-proBNP was markedly elevated during the clinical course."
    },

    # ---------- Treatment ----------
    {
        "id": "F08",
        "category": "treatment",
        "fact": "Supplemental low-flow oxygen therapy was used to treat hypoxia."
    },
    {
        "id": "F09",
        "category": "treatment",
        "fact": "Therapeutic anticoagulation was initiated during the clinical course."
    },
    {
        "id": "F10",
        "category": "treatment",
        "fact": "Rivaroxaban was used for ongoing/long-term anticoagulation."
    },
    {
        "id": "F11",
        "category": "treatment",
        "fact": "Furosemide was used during management of pulmonary hypertension/right-heart strain and fluid retention."
    },
    {
        "id": "F12",
        "category": "treatment",
        "fact": "Sildenafil 20 mg three times daily was introduced for pulmonary hypertension management."
    },
    {
        "id": "F13",
        "category": "treatment",
        "fact": "Physiotherapy included breathing exercises and progressive mobilisation."
    },
    {
        "id": "F14",
        "category": "treatment",
        "fact": "Energy-conservation/pacing strategies and a home exercise programme were provided."
    },
    {
        "id": "F15",
        "category": "treatment",
        "fact": "Mild anaemia was documented later in the course and treated with ferrous sulfate."
    },

    # ---------- Clinical progression ----------
    {
        "id": "F16",
        "category": "progression",
        "fact": "Breathlessness and exercise tolerance improved during the clinical course."
    },
    {
        "id": "F17",
        "category": "progression",
        "fact": "Supplemental oxygen requirements decreased, with the patient later maintaining approximately 94-95% SpO2 on room air."
    },

    # ---------- Referral / outcome ----------
    {
        "id": "F18",
        "category": "outcome",
        "fact": "The patient was referred to a tertiary pulmonary hypertension centre for pulmonary endarterectomy assessment."
    },
    {
        "id": "F19",
        "category": "outcome",
        "fact": "The patient was ultimately discharged after clinical improvement."
    },
    {
        "id": "F20",
        "category": "outcome",
        "fact": "Follow-up at the tertiary pulmonary hypertension centre was planned after discharge."
    },
]

In [49]:
print("Reference facts:", len(reference_facts))

for item in reference_facts:
    print(
        f"{item['id']} | {item['fact']}"
    )

Reference facts: 20
F01 | Patient presented with progressive exertional breathlessness.
F02 | Patient was hypoxic at presentation, with SpO2 89% on room air.
F03 | Chronic thromboembolic pulmonary hypertension (CTEPH) was diagnosed.
F04 | CT pulmonary angiography confirmed chronic thromboembolic disease.
F05 | Echocardiography demonstrated severe pulmonary hypertension with right ventricular strain.
F06 | V/Q scanning demonstrated mismatched perfusion defects consistent with CTEPH.
F07 | NT-proBNP was markedly elevated during the clinical course.
F08 | Supplemental low-flow oxygen therapy was used to treat hypoxia.
F09 | Therapeutic anticoagulation was initiated during the clinical course.
F10 | Rivaroxaban was used for ongoing/long-term anticoagulation.
F11 | Furosemide was used during management of pulmonary hypertension/right-heart strain and fluid retention.
F12 | Sildenafil 20 mg three times daily was introduced for pulmonary hypertension management.
F13 | Physiotherapy included b

In [16]:
# ============================================================
# REFERENCE-FACT COVERAGE JUDGE PROMPT
# ============================================================
#
# The evaluator receives:
#   1. The complete frozen reference-fact checklist.
#   2. One generated RAG summary.
#
# It independently assigns a 0/1/2 coverage score to every
# reference fact.
#
# The prompt explicitly separates coverage from faithfulness:
# unsupported information is NOT penalized in this evaluation.
# ============================================================

REFERENCE_COVERAGE_PROMPT = """
You are evaluating the information coverage of a generated clinical summary.

You will receive:
1. A list of reference clinical facts derived from the source clinical record.
2. A generated clinical summary.

For EACH reference fact, determine how completely the clinical information
in that reference fact is represented in the generated summary.

Scoring:
2 = FULLY CAPTURED
    The clinically important meaning of the reference fact is clearly
    represented in the generated summary. Exact wording is not required.

1 = PARTIALLY CAPTURED
    The core clinical concept is represented, but clinically meaningful
    information contained in the reference fact is missing or incomplete.

0 = ABSENT
    The clinical information in the reference fact is not represented
    in the generated summary.

Important rules:
- Evaluate semantic meaning, not exact word overlap.
- Do not reward information merely because it is medically related.
- Do not infer information that the generated summary does not state.
- Score each reference fact independently.
- Do not evaluate whether the generated summary is factually correct here.
  This evaluation measures COVERAGE only.
- Do not penalize additional information in the generated summary.
  Unsupported information will be evaluated separately.
- For every score, provide a short evidence phrase copied or closely
  paraphrased from the generated summary.
- If the score is 0, write "Not represented" as the evidence.

Return ONLY valid JSON in this exact structure:

{
  "scores": [
    {
      "fact_id": "F01",
      "score": 0,
      "evidence": "..."
    }
  ]
}
"""

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

In [25]:
from openai import OpenAI

from src.llm.llm import LLM_MODEL, LLM_PROVIDER

coverage_client = OpenAI(
    api_key= os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = LLM_MODEL
EVALUATOR_PROVIDER = LLM_PROVIDER

print("Coverage evaluator provider:", EVALUATOR_PROVIDER)
print("Coverage evaluator model:", EVALUATOR_MODEL)

Coverage evaluator provider: OpenAI
Coverage evaluator model: gpt-5.6-luna


In [28]:
def evaluate_reference_coverage(
    summary: str,
    reference_facts: list[dict],
) -> list[dict]:
    """
    Evaluate one generated clinical summary against the frozen
    reference-fact checklist.

    Each fact receives:
        2 = fully captured
        1 = partially captured
        0 = absent

    Structured output is enforced so the response can be parsed
    reliably without manual JSON cleanup.
    """

    facts_text = "\n".join(
        f"{item['id']}: {item['fact']}"
        for item in reference_facts
    )

    user_message = f"""
REFERENCE FACTS:
{facts_text}

GENERATED SUMMARY:
{summary}
"""

    response = coverage_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": REFERENCE_COVERAGE_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "reference_coverage",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "scores": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "fact_id": {
                                        "type": "string"
                                    },
                                    "score": {
                                        "type": "integer",
                                        "enum": [0, 1, 2],
                                    },
                                    "evidence": {
                                        "type": "string"
                                    },
                                },
                                "required": [
                                    "fact_id",
                                    "score",
                                    "evidence",
                                ],
                                "additionalProperties": False,
                            },
                        }
                    },
                    "required": ["scores"],
                    "additionalProperties": False,
                },
            },
        },
    )

    raw_output = response.choices[0].message.content
    result = json.loads(raw_output)

    return result["scores"]

In [29]:
# ============================================================
# SINGLE-CONFIGURATION SANITY TEST
# ============================================================
#
# Run the evaluator on one configuration before evaluating all
# 9 summaries.
#
# We verify that:
# - valid JSON is returned
# - every frozen reference fact receives a score
# - scores are limited to 0, 1, or 2
# ============================================================

test_configuration = "Whole + Gemini"

test_summary = rag_results[
    test_configuration
]["summary"]

test_coverage = evaluate_reference_coverage(
    summary=test_summary,
    reference_facts=reference_facts,
)

print("Configuration:", test_configuration)
print("Reference facts:", len(reference_facts))
print("Returned scores:", len(test_coverage))

assert len(test_coverage) == len(reference_facts)

assert all(
    item["score"] in {0, 1, 2}
    for item in test_coverage
)

print("Coverage sanity check passed.")

Configuration: Whole + Gemini
Reference facts: 20
Returned scores: 20
Coverage sanity check passed.


In [30]:
# ============================================================
# RUN REFERENCE-FACT COVERAGE — ALL 9 CONFIGURATIONS
# ============================================================
#
# COST / RELIABILITY CONTROLS
# ---------------------------
# - One evaluator call is required per configuration.
# - The already completed Whole + Gemini sanity-test result
#   is reused rather than evaluated again.
# - Each successful result is checkpointed immediately.
# - If this cell is rerun, completed configurations are skipped.
# - Execution stops on an API/parsing error rather than
#   automatically retrying and creating unnecessary calls.
#
# The generated RAG summaries and frozen reference facts are
# NOT modified during this evaluation.
# ============================================================

In [31]:
COVERAGE_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_reference_fact_coverage_gpt56luna.json"
)

COVERAGE_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Load previously completed evaluations if they exist.
if COVERAGE_OUTPUT_PATH.exists():
    with COVERAGE_OUTPUT_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        coverage_results = json.load(file)
else:
    coverage_results = {}

In [33]:
# Reuse the successful sanity-test evaluation.
if test_configuration not in coverage_results:
    coverage_results[test_configuration] = test_coverage

    with COVERAGE_OUTPUT_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            coverage_results,
            file,
            indent=2,
            ensure_ascii=False,
        )


In [34]:
import time

for configuration, result in rag_results.items():

    # Avoid paying for an evaluation that already exists.
    if configuration in coverage_results:
        print(f"Skipping completed: {configuration}")
        continue

    print(f"\nEvaluating coverage: {configuration}")

    try:
        scores = evaluate_reference_coverage(
            summary=result["summary"],
            reference_facts=reference_facts,
        )

        # Validate the evaluator output before saving it.
        assert len(scores) == len(reference_facts)

        assert all(
            item["score"] in {0, 1, 2}
            for item in scores
        )

        coverage_results[configuration] = scores

        # Checkpoint immediately after each successful call.
        with COVERAGE_OUTPUT_PATH.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                coverage_results,
                file,
                indent=2,
                ensure_ascii=False,
            )

        print(f"Saved: {configuration}")

    except Exception as error:
        print(
            f"FAILED: {configuration}\n"
            f"{type(error).__name__}: {error}"
        )
        break

    time.sleep(2)


print(
    f"\nCompleted coverage evaluations: "
    f"{len(coverage_results)}/"
    f"{len(rag_results)}"
)

print("Checkpoint:", COVERAGE_OUTPUT_PATH)

Skipping completed: Whole + Gemini

Evaluating coverage: Whole + BGE
Saved: Whole + BGE

Evaluating coverage: Whole + MedCPT
Saved: Whole + MedCPT

Evaluating coverage: Fixed + Gemini
Saved: Fixed + Gemini

Evaluating coverage: Fixed + BGE
Saved: Fixed + BGE

Evaluating coverage: Fixed + MedCPT
Saved: Fixed + MedCPT

Evaluating coverage: Section + Gemini
Saved: Section + Gemini

Evaluating coverage: Section + BGE
Saved: Section + BGE

Evaluating coverage: Section + MedCPT
Saved: Section + MedCPT

Completed coverage evaluations: 9/9
Checkpoint: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_reference_fact_coverage_gpt56luna.json


In [35]:
# ============================================================
# COVERAGE RESULTS SUMMARY
# ============================================================
#
# The LLM-based coverage evaluation is now COMPLETE.
#
# This section performs NO LLM calls.
# It only aggregates the previously saved 0/1/2 judgments
# into configuration-level coverage statistics.
#
# For 20 reference facts:
#     Maximum score = 20 x 2 = 40 points
#
# Reported metrics:
# - Coverage Points
# - Coverage %
# - Fully Captured facts
# - Partially Captured facts
# - Absent facts
# ============================================================

coverage_summary = []

max_score = len(reference_facts) * 2

for configuration, scores in coverage_results.items():

    score_values = np.array([
        item["score"]
        for item in scores
    ])

    total_score = score_values.sum()

    coverage_summary.append(
        {
            "Configuration": configuration,
            "Coverage Points": int(total_score),
            "Max Points": max_score,
            "Coverage %": round(
                (total_score / max_score) * 100,
                1,
            ),
            "Fully Captured": int(
                (score_values == 2).sum()
            ),
            "Partially Captured": int(
                (score_values == 1).sum()
            ),
            "Absent": int(
                (score_values == 0).sum()
            ),
        }
    )


coverage_summary_df = pd.DataFrame(
    coverage_summary
)

coverage_summary_df = (
    coverage_summary_df
    .sort_values(
        "Coverage %",
        ascending=False,
    )
    .reset_index(drop=True)
)

coverage_summary_df

,Configuration,Coverage Points,Max Points,Coverage %,Fully Captured,Partially Captured,Absent
0,Whole + Gemini,40,40,100.0,20,0,0
1,Whole + BGE,39,40,97.5,19,1,0
2,Fixed + BGE,39,40,97.5,19,1,0
3,Whole + MedCPT,38,40,95.0,18,2,0
4,Fixed + Gemini,37,40,92.5,17,3,0
5,Fixed + MedCPT,36,40,90.0,17,2,1
6,Section + BGE,35,40,87.5,16,3,1
7,Section + MedCPT,29,40,72.5,12,5,3
8,Section + Gemini,21,40,52.5,9,3,8


In [36]:
# ============================================================
# COVERAGE RESULT — INITIAL OBSERVATION
# ============================================================
#
# Whole-note configurations achieved the strongest reference-
# fact coverage overall. Whole + Gemini captured all 20 frozen
# reference facts (100%), while Whole + BGE and Fixed + BGE
# each achieved 97.5%.
#
# Section-aware configurations showed lower coverage,
# particularly Section + Gemini (52.5%). Because retrieval was
# fixed at Top-20 chunks, finer-grained section chunking also
# resulted in substantially smaller retrieved contexts.
#
# These results measure information coverage ONLY and should
# not be interpreted as the final configuration ranking.
# Faithfulness, temporal consistency, and other evaluation
# dimensions are assessed separately.
# ============================================================

In [50]:
# ============================================================
# EVALUATION 2 — TF-IDF COSINE SIMILARITY
# ============================================================
#
# PURPOSE
# -------
# Measure lexical similarity between each generated RAG summary
# and the complete source clinical record.
#
# For each configuration:
#
#   1. Fit TF-IDF jointly on:
#        - the complete source clinical record
#        - the generated summary
#
#   2. Represent both texts in the same TF-IDF vector space.
#
#   3. Calculate cosine similarity between the source-record
#      vector and the generated-summary vector.
#
#
# INTERPRETATION
# --------------
# Higher similarity indicates greater lexical/content overlap
# with the complete source record.
#
# This metric does NOT directly measure:
# - clinical factuality
# - information importance
# - temporal correctness
# - semantic equivalence when different terminology is used
#
# It therefore complements, rather than replaces, the
# reference-fact coverage evaluation.
#
#
# IMPORTANT
# ---------
# This evaluation is deterministic and makes NO LLM/API calls.
# ============================================================

In [54]:
NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes = pd.read_csv(NOTES_PATH)

In [74]:
PATIENT_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

In [77]:
# ------------------------------------------------------------
# Clean and deduplicate the source notes using the same
# preprocessing applied in the RAG configuration experiment.
# ------------------------------------------------------------

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first",
    )
    .reset_index(drop=True)
)

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Patient:", PATIENT_ID)
print("Patient notes:", len(patient_notes))

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Patient notes: 45


In [78]:
# ------------------------------------------------------------
# Build the complete source record for TF-IDF evaluation.
#
# Use the same 45 cleaned and deduplicated patient notes used
# in the RAG configuration experiment.
# ------------------------------------------------------------

source_text = "\n\n".join(
    patient_notes["clean_note_text"]
    .astype(str)
    .tolist()
)

print("Patient:", PATIENT_ID)
print("Source notes:", len(patient_notes))
print("Source characters:", len(source_text))
print("Source words:", len(source_text.split()))

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Source notes: 45
Source characters: 35863
Source words: 5134


In [58]:
# ============================================================
# BUILD COMPLETE SOURCE RECORD FOR TF-IDF EVALUATION
# ============================================================
#
# TF-IDF compares each generated RAG summary against the
# COMPLETE clinical record for the evaluation patient.
#
# IMPORTANT:
# - This is NOT a retrieved Top-20 RAG context.
# - It contains all clinical notes available for this patient.
# - Notes are ordered chronologically before concatenation.
# - This same source record is used for all 9 configurations.
#
# No LLM/API calls are made here.
# ============================================================

In [79]:
tfidf_results = []

for configuration, result in rag_results.items():

    summary = result["summary"]

    # Fit TF-IDF jointly on the complete source record and
    # this configuration's generated summary.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
    )

    vectors = vectorizer.fit_transform(
        [
            source_text,
            summary,
        ]
    )

    # Compare the complete source-record vector with the
    # generated-summary vector.
    similarity = cosine_similarity(
        vectors[0:1],
        vectors[1:2],
    )[0][0]

    tfidf_results.append(
        {
            "Configuration": configuration,
            "TF-IDF Cosine Similarity": similarity,
        }
    )


tfidf_similarity_df = pd.DataFrame(
    tfidf_results
)

tfidf_similarity_df = (
    tfidf_similarity_df
    .sort_values(
        "TF-IDF Cosine Similarity",
        ascending=False,
    )
    .reset_index(drop=True)
)

tfidf_similarity_df

,Configuration,TF-IDF Cosine Similarity
0,Whole + MedCPT,0.396356
1,Section + BGE,0.386688
2,Whole + BGE,0.377191
3,Fixed + MedCPT,0.373188
4,Fixed + Gemini,0.370844
5,Whole + Gemini,0.367396
6,Fixed + BGE,0.367226
7,Section + MedCPT,0.347418
8,Section + Gemini,0.257903


In [80]:
print("Source characters:", len(source_text))
print("Source words:", len(source_text.split()))

Source characters: 35863
Source words: 5134


In [81]:
# ============================================================
# SAVE TF-IDF EVALUATION RESULTS
# ============================================================
#
# Evaluation 2 is complete.
#
# These scores were calculated deterministically and required
# no LLM/API calls.
#
# The output is saved separately so downstream ranking and
# analysis can reuse it without recomputing the evaluation.
# ============================================================

TFIDF_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_tfidf_similarity_gpt56luna_top20.csv"
)

tfidf_similarity_df.to_csv(
    TFIDF_OUTPUT_PATH,
    index=False,
)

print("Saved:", TFIDF_OUTPUT_PATH)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag_tfidf_similarity_gpt56luna_top20.csv


In [64]:
# ============================================================
# TEMPORAL EVALUATION MODEL
# ============================================================
#
# GPT-5.6 Luna is used as the LLM judge for semantic event
# alignment in the regenerated evaluation pipeline.
#
# The summaries being evaluated were also generated with
# GPT-5.6 Luna; however, generation and evaluation are separate
# API calls with separate prompts and purposes.
#
# The consolidated reference timeline is reused from the
# original experiment because it was derived exclusively from
# the unchanged source clinical record, not from generated
# summaries.
# ============================================================

In [63]:
# ============================================================
# EVALUATION 4 — TEMPORAL CONSISTENCY
#
# Evaluates whether clinical events appear in the generated
# summaries in the same relative order as the source record.
#
# This evaluation is rerun against the newly regenerated
# GPT-5.6-Luna RAG summaries.
# ============================================================

EVALUATOR_MODEL = "gpt-5.6-luna"

rag_summaries = {
    configuration: result["summary"]
    for configuration, result in rag_results.items()
}

print("Evaluator model:", EVALUATOR_MODEL)
print("RAG summaries:", len(rag_summaries))

Evaluator model: gpt-5.6-luna
RAG summaries: 9


In [ ]:
# ------------------------------------------------------------
# build the exact 45-note patient record used in the
# configuration experiment.
# ------------------------------------------------------------

SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

notes = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"]
        == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("patient_notes:", len(patient_notes))

In [68]:
# ------------------------------------------------------------
# Prepare the 45 source notes for temporal evaluation.
#
# source_note_index gives each note its authoritative
# chronological position based on creation_timestamp.
# ------------------------------------------------------------

temporal_notes = (
    patient_notes
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
    .copy()
)

temporal_notes["source_note_index"] = (
    temporal_notes.index
)

print("Temporal source notes:", len(temporal_notes))

temporal_notes[
    [
        "source_note_index",
        "creation_timestamp",
        "clean_note_text",
    ]
].head()

Temporal source notes: 90


,source_note_index,creation_timestamp,clean_note_text
0,0,03/01/2026 01:35,"Patient Abena Bonsu, a 43-year-old female, was..."
1,1,03/01/2026 01:35,"Patient Abena Bonsu, a 43-year-old female, was..."
2,2,03/01/2026 02:15,Patient presented with progressive breathlessn...
3,3,03/01/2026 02:15,Patient presented with progressive breathlessn...
4,4,03/01/2026 02:45,Date: 03/01/26\nTime: 02:45\nStaff: Nurse Audr...


In [69]:
# ---------------------------------------------------------
# EVALUATION 4 — TEMPORAL CONSISTENCY
# Step 3A: Extract longitudinal clinical events
#
# Each source note is processed separately so that every
# extracted event remains tied to its chronological position.
# ---------------------------------------------------------

TEMPORAL_EVENT_PROMPT = """
Extract the clinically meaningful CURRENT events from this clinical note
for longitudinal temporal evaluation.

The source notes have already been ordered using creation_timestamp.
Do NOT use dates written inside the note to establish chronology.

Extract events that occur, are newly observed, are newly decided, or
represent a change in clinical state at THIS point in the record.

Include:
- new clinical findings or changes in patient status
- investigations ordered, performed, or newly reviewed
- new diagnoses or diagnostic conclusions
- treatments started, stopped, changed, or continued when clinically meaningful
- referrals or disposition decisions
- meaningful rehabilitation or management progression

Do NOT extract:
- background medical history or allergies
- staff/administrative information
- repeated historical information merely restated from earlier care
- earlier events mentioned only as context
- dates or times as separate events
- duplicate statements of the same event within the note

Distinguish plans from completed actions.
For example, "admission planned" must remain a plan and must not become
"patient admitted."

Use only information explicitly stated in the note.
Keep each event concise and patient-specific.

Return valid JSON only:

{
  "events": [
    "event 1",
    "event 2"
  ]
}
"""

In [71]:
def extract_temporal_events(note_text):
    """
    Extract candidate temporal clinical events from one source note.
    """

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_EVENT_PROMPT,
            },
            {
                "role": "user",
                "content": note_text,
            },
        ],
    )

    raw_output = response.choices[0].message.content

    try:
        result = json.loads(raw_output)
        return result["events"]

    except json.JSONDecodeError:
        print("Could not parse JSON:")
        print(raw_output)
        return []

In [72]:
    # ------------------------------------------------------------
    # Extract candidate temporal events from all 45 source notes.
    #
    # Each note is processed independently so candidate events retain
    # their authoritative source_note_index and creation_timestamp.
    # Extraction failures are recorded separately rather than dropped
    # silently.
    # ------------------------------------------------------------

    temporal_events = []
    failed_notes = []

    for i, row in temporal_notes.iterrows():

        events = extract_temporal_events(
            row["clean_note_text"]
        )

        if events is None:
            failed_notes.append(
                int(row["source_note_index"])
            )

            print(
                f"{i + 1:02d}/45 | "
                f"NOTE {row['source_note_index']} | FAILED"
            )
            continue

        for event in events:
            temporal_events.append(
                {
                    "source_note_index": int(
                        row["source_note_index"]
                    ),
                    "creation_timestamp": row[
                        "creation_timestamp"
                    ],
                    "event": event,
                }
            )

        print(
            f"{i + 1:02d}/45 | "
            f"NOTE {row['source_note_index']} | "
            f"{len(events)} events"
        )

    print("\nTotal candidate events:", len(temporal_events))
    print("Failed notes:", failed_notes)

01/45 | NOTE 0 | 4 events
02/45 | NOTE 1 | 4 events
03/45 | NOTE 2 | 4 events
04/45 | NOTE 3 | 4 events
05/45 | NOTE 4 | 3 events
06/45 | NOTE 5 | 4 events
07/45 | NOTE 6 | 5 events
08/45 | NOTE 7 | 4 events
09/45 | NOTE 8 | 12 events
10/45 | NOTE 9 | 11 events
11/45 | NOTE 10 | 9 events
12/45 | NOTE 11 | 10 events
13/45 | NOTE 12 | 11 events
14/45 | NOTE 13 | 6 events
15/45 | NOTE 14 | 4 events
16/45 | NOTE 15 | 4 events
17/45 | NOTE 16 | 3 events
18/45 | NOTE 17 | 3 events
19/45 | NOTE 18 | 5 events
20/45 | NOTE 19 | 6 events
21/45 | NOTE 20 | 7 events
22/45 | NOTE 21 | 6 events
23/45 | NOTE 22 | 3 events
24/45 | NOTE 23 | 3 events
25/45 | NOTE 24 | 6 events
26/45 | NOTE 25 | 7 events
27/45 | NOTE 26 | 5 events
28/45 | NOTE 27 | 5 events
29/45 | NOTE 28 | 3 events
30/45 | NOTE 29 | 3 events
31/45 | NOTE 30 | 5 events
32/45 | NOTE 31 | 5 events
33/45 | NOTE 32 | 7 events
34/45 | NOTE 33 | 7 events
35/45 | NOTE 34 | 8 events
36/45 | NOTE 35 | 8 events
37/45 | NOTE 36 | 7 events
38/45 |

KeyboardInterrupt: 

In [73]:
print("patient_notes:", len(patient_notes))
print("temporal_notes:", len(temporal_notes))

print(
    "Unique temporal source indices:",
    temporal_notes["source_note_index"].nunique()
)

print(
    "Index range:",
    temporal_notes["source_note_index"].min(),
    "to",
    temporal_notes["source_note_index"].max(),
)

patient_notes: 90
temporal_notes: 90
Unique temporal source indices: 90
Index range: 0 to 89
